In [3]:
!pip install -q transformers datasets accelerate

In [1]:
# --- Quick check before writing cell 1 ---
import os
from pathlib import Path

# 1. Is Drive mounted?
drive_root = Path("/content/drive/MyDrive")
print(f"Drive mounted    : {drive_root.exists()}")

# 2. Does the folder exist?
splits_dir = drive_root / "persian_nlp_pipline"
print(f"Splits folder    : {splits_dir.exists()}")

# 3. List files in that folder
if splits_dir.exists():
    for f in sorted(splits_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name:<30} {size_mb:>8.1f} MB")

# 4. GPU check
import torch
print()
print(f"torch version    : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}")

# 5. transformers version
import transformers
print()
print(f"transformers ver : {transformers.__version__}")

Drive mounted    : False
Splits folder    : False

torch version    : 2.14.1+cu130
CUDA available   : True
GPU              : Tesla T4
GPU memory (GB)  : 14.6

transformers ver : 5.22.1


In [3]:
from pathlib import Path
drive_root = Path("/content/drive/MyDrive")
splits_dir = drive_root / "persian_nlp_pipline"
print(f"Drive mounted    : {drive_root.exists()}")
print(f"Splits folder    : {splits_dir.exists()}")
if splits_dir.exists():
    for f in sorted(splits_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name:<30} {size_mb:>8.1f} MB")

Drive mounted    : True
Splits folder    : True
  test.csv                          118.0 MB
  train_supervised.csv              458.1 MB
  validation.csv                    117.4 MB


In [9]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 1: Setup
# ============================================================
# Environment: Google Colab, NVIDIA T4 (14.6 GB), Drive mounted.
#
# Goal:
#   - mount Drive (if not already)
#   - verify required packages
#   - fail-fast if no GPU
#   - define paths, seeds, hyperparameters in one place
#
# No files are written here. Models are saved in cell 7.
# ============================================================

# --- 1. Mount Drive (idempotent: safe to re-run) -------------------------
from pathlib import Path
import sys, os, json, random, datetime

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.exists(), "Google Drive is not mounted."

# --- 2. Verify required packages ----------------------------------------
REQUIRED = {
    "torch": None,
    "transformers": None,
    "datasets": None,
    "accelerate": None,
    "sklearn": "sklearn",
    "polars": None,
    "numpy": None,
}
missing = []
for import_name, pip_name in REQUIRED.items():
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"[ok] {import_name:<14} {version}")
    except ImportError:
        print(f"[MISSING] {import_name}")
        missing.append(pip_name or import_name)

if missing:
    print(f"\nInstalling missing packages: {missing}")
    import subprocess
    subprocess.run(["pip", "install", "-q", *missing], check=True)
    print("Installed. Please re-run this cell to verify.")
    raise SystemExit("Re-run cell after installing packages.")

# --- 3. GPU check (fail-fast) -------------------------------------------
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. This notebook is not practical on CPU. "
    "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU."
)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print()
print(f"GPU              : {gpu_name}")
print(f"GPU memory (GB)  : {gpu_mem_gb:.1f}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"PyTorch version  : {torch.__version__}")

# --- 4. Version compatibility check -------------------------------------
import transformers
print(f"transformers ver : {transformers.__version__}")

_tf_major = int(transformers.__version__.split(".")[0])
if _tf_major >= 5:
    print("  NOTE: transformers >= 5 detected. "
          "Some TrainingArguments defaults changed; "
          "we pass explicit values for everything we rely on.")

# --- 5. Paths -----------------------------------------------------------
SPLITS_DIR = DRIVE_ROOT / "persian_nlp_pipline"

TRAIN_CSV = SPLITS_DIR / "train_supervised.csv"
VAL_CSV   = SPLITS_DIR / "validation.csv"
TEST_CSV  = SPLITS_DIR / "test.csv"

for _p in (TRAIN_CSV, VAL_CSV, TEST_CSV):
    assert _p.exists(), f"Missing split file: {_p}"

# Output directories (created here, written by later cells)
OUTPUT_ROOT = DRIVE_ROOT / "persian_nlp_pipline" / "artifacts"
REC_OUTPUT_DIR  = OUTPUT_ROOT / "bert_recommendation"
BAND_OUTPUT_DIR = OUTPUT_ROOT / "bert_rating_band"
for _d in (REC_OUTPUT_DIR, BAND_OUTPUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

METRICS_JSON = OUTPUT_ROOT / "transformer_metrics.json"

# --- 6. Seeds (one source of truth, chains Part A/B/C) ------------------
SPLIT_SEED = 2026
SUBSET_SEED = SPLIT_SEED
TRAIN_SEED = SPLIT_SEED

random.seed(TRAIN_SEED)
import numpy as np
np.random.seed(TRAIN_SEED)
torch.manual_seed(TRAIN_SEED)
torch.cuda.manual_seed_all(TRAIN_SEED)

# --- 7. Model & training hyperparameters --------------------------------
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"  # ParsBERT v2.0
MAX_LENGTH = 128
SUBSET_SIZE = 500_000

# Training (per model)
BATCH_SIZE = 16
GRAD_ACCUM = 2          # effective batch size = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
FP16 = True
EARLY_STOP_PATIENCE = 1
EVAL_STEPS = 2000        # evaluate every N optimizer steps
SAVE_STEPS = 2000      # checkpoint every N optimizer steps

# Labels
REC_LABELS  = ["recommended", "not_recommended", "no_idea"]
BAND_LABELS = ["low", "medium", "high"]

# --- 8. Ready message ---------------------------------------------------
print()
print("=" * 60)
print("Notebook 04 — environment ready.")
print("=" * 60)
print(f"SPLITS_DIR       : {SPLITS_DIR}")
print(f"TRAIN_CSV        : {TRAIN_CSV.name}  "
      f"({TRAIN_CSV.stat().st_size / 1024**2:.1f} MB)")
print(f"VAL_CSV          : {VAL_CSV.name}  "
      f"({VAL_CSV.stat().st_size / 1024**2:.1f} MB)")
print(f"TEST_CSV         : {TEST_CSV.name}  "
      f"({TEST_CSV.stat().st_size / 1024**2:.1f} MB)")
print()
print(f"OUTPUT_ROOT      : {OUTPUT_ROOT}")
print(f"REC_OUTPUT_DIR   : {REC_OUTPUT_DIR.name}/")
print(f"BAND_OUTPUT_DIR  : {BAND_OUTPUT_DIR.name}/")
print(f"METRICS_JSON     : {METRICS_JSON.name}")
print()
print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"MAX_LENGTH       : {MAX_LENGTH}")
print(f"SUBSET_SIZE      : {SUBSET_SIZE:,}")
print(f"BATCH_SIZE       : {BATCH_SIZE}  (x{GRAD_ACCUM} accum = {BATCH_SIZE * GRAD_ACCUM})")
print(f"EPOCHS           : {EPOCHS}")
print(f"LEARNING_RATE    : {LEARNING_RATE}")
print(f"FP16             : {FP16}")
print(f"EARLY_STOP       : patience={EARLY_STOP_PATIENCE}")
print()
print(f"SPLIT_SEED       : {SPLIT_SEED}")

[ok] torch          2.14.1+cu130
[ok] transformers   5.22.1
[ok] datasets       4.8.5
[ok] accelerate     1.15.0
[ok] sklearn        1.6.1
[ok] polars         1.35.2
[ok] numpy          2.4.6

GPU              : Tesla T4
GPU memory (GB)  : 14.6
CUDA version     : 13.0
PyTorch version  : 2.14.1+cu130
transformers ver : 5.22.1
  NOTE: transformers >= 5 detected. Some TrainingArguments defaults changed; we pass explicit values for everything we rely on.

Notebook 04 — environment ready.
SPLITS_DIR       : /content/drive/MyDrive/persian_nlp_pipline
TRAIN_CSV        : train_supervised.csv  (458.1 MB)
VAL_CSV          : validation.csv  (117.4 MB)
TEST_CSV         : test.csv  (118.0 MB)

OUTPUT_ROOT      : /content/drive/MyDrive/persian_nlp_pipline/artifacts
REC_OUTPUT_DIR   : bert_recommendation/
BAND_OUTPUT_DIR  : bert_rating_band/
METRICS_JSON     : transformer_metrics.json

MODEL_NAME       : HooshvareLab/bert-fa-base-uncased
MAX_LENGTH       : 128
SUBSET_SIZE      : 500,000
BATCH_SIZE   

In [10]:
import json
from pathlib import Path

manifest_path = Path("/content/drive/MyDrive/persian_nlp_pipline/split_manifest.json")
assert manifest_path.exists(), "manifest not uploaded yet"

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

print("Seeds        :", manifest["seeds"])
print("Counts       :", manifest["counts"])
print("Band thresh  :", manifest["rating_band"]["thresholds"])
print("Ordinal enc  :", manifest["rating_band"]["ordinal_encoding"])
print("Masking      :", manifest["masking"]["per_class"])

Seeds        : {'SPLIT_SEED': 2026, 'MASK_SEED': 'digikala-semi-supervised-v1', 'MASK_RATE': 0.2}
Counts       : {'total_clean_rows': 6149834, 'products': 331512, 'train_products': 232057, 'val_products': 49727, 'test_products': 49728, 'train_supervised': 3578692, 'train_pseudo_unlabeled': 737731, 'validation': 913570, 'test': 919841, 'natural_unlabeled': 627763}
Band thresh  : {'low': '0 <= rate < 2.5', 'medium': '2.5 <= rate < 4.0', 'high': '4.0 <= rate <= 5.0', 'invalid': 'rate missing or outside [0, 5]'}
Ordinal enc  : {'low': 0, 'medium': 1, 'high': 2}
Masking      : {'no_idea': {'total': 415723, 'masked': 83144, 'mask_ratio_actual': 0.199999}, 'not_recommended': {'total': 351901, 'masked': 70380, 'mask_ratio_actual': 0.199999}, 'recommended': {'total': 2921036, 'masked': 584207, 'mask_ratio_actual': 0.2}}


In [11]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 2: Load & Stratified Subset
# ============================================================
# Goal:
#   1. (optional) verify counts against split_manifest.json
#   2. load the three split CSVs
#   3. build a 500K stratified subset of train_supervised
#      by joint stratification on (recommendation_status, rating_band)
#   4. write /content/train_subset.parquet for fast reload
#
# Writes: /content/train_subset.parquet (temporary, not in Drive)
# ============================================================

import polars as pl
import math
import time
from pathlib import Path

# --- 1. (Optional) verify against manifest if it was uploaded -----------
manifest_path = SPLITS_DIR / "split_manifest.json"
if manifest_path.exists():
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    print("Manifest found — verifying row counts.")
    print(f"  expected train_supervised : {manifest['counts']['train_supervised']:,}")
    print(f"  expected validation       : {manifest['counts']['validation']:,}")
    print(f"  expected test             : {manifest['counts']['test']:,}")
    print()
else:
    manifest = None
    print("No split_manifest.json found — skipping count verification.")
    print()

# --- 2. Load the three split CSVs ---------------------------------------
# We only need the columns used for training and for stratification.
NEEDED_COLS = [
    "id",
    "body_normalized",
    "recommendation_status",
    "rating_band",
    "is_valid_rating_band",
]

print("Loading split CSVs with polars...")
t0 = time.time()
train_full = pl.read_csv(
    TRAIN_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

val_full = pl.read_csv(
    VAL_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

test_full = pl.read_csv(
    TEST_CSV, separator=",", has_header=True,
    null_values=["", "nan", "NaN", "null", "None"],
    infer_schema_length=10_000,
).select(NEEDED_COLS)

t_load = time.time() - t0
print(f"  loaded in {t_load:.1f}s")
print(f"  train : {train_full.height:,}")
print(f"  val   : {val_full.height:,}")
print(f"  test  : {test_full.height:,}")

# Verify if manifest was present
if manifest is not None:
    assert train_full.height == manifest["counts"]["train_supervised"], "train count mismatch"
    assert val_full.height   == manifest["counts"]["validation"],       "val count mismatch"
    assert test_full.height  == manifest["counts"]["test"],             "test count mismatch"
    print("  counts match manifest. OK.")
print()

# --- 3. Keep only rows usable for joint supervision ---------------------
# Rationale: this subset is shared by Task A and Task B.
# Rows must have both labels. That costs us the ~627K rows that only have
# a rating_band label but no recommendation_status. For a shared subset,
# that is the right trade-off.
print("Filtering to rows with both labels present...")
t0 = time.time()
train_usable = train_full.filter(
    pl.col("recommendation_status").is_not_null()
    & pl.col("rating_band").is_not_null()
    & (pl.col("rating_band") != "invalid")
    & pl.col("body_normalized").is_not_null()
)
t_filter = time.time() - t0
print(f"  filtered in {t_filter:.1f}s")
print(f"  usable train rows : {train_usable.height:,}  "
      f"({100 * train_usable.height / train_full.height:.2f}% of train)")
print()

# --- 4. Joint stratification on (recommendation_status, rating_band) ----
# For each (rec, band) group, sample n_group = floor(SUBSET_SIZE * n_group / N).
# This preserves the marginal distribution of BOTH targets as closely as
# possible given the 500K budget.
print(f"Building {SUBSET_SIZE:,}-row stratified subset (seed={SUBSET_SEED})...")
t0 = time.time()

# Add a joint group key and compute counts
train_usable = train_usable.with_columns(
    (pl.col("recommendation_status") + "|" + pl.col("rating_band")).alias("_group")
)

group_counts = (
    train_usable
    .group_by("_group")
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * SUBSET_SIZE)
        .floor()
        .cast(pl.Int64)
        .alias("n_sample")
    )
    .sort("_group")
)
print("  Joint group sizes and sample targets:")
print(group_counts)
print()

# Sample per group
sampled_parts = []
for row in group_counts.iter_rows(named=True):
    grp = row["_group"]
    n_take = row["n_sample"]
    if n_take <= 0:
        continue
    df_grp = train_usable.filter(pl.col("_group") == grp)
    part = df_grp.sample(n=n_take, seed=SUBSET_SEED, with_replacement=False)
    sampled_parts.append(part)

train_subset = pl.concat(sampled_parts).drop("_group")

# Because of floor(), we may be slightly under SUBSET_SIZE.
# Top up from the largest group if needed.
shortfall = SUBSET_SIZE - train_subset.height
if shortfall > 0:
    print(f"  topping up {shortfall:,} rows from the largest group...")
    biggest_group = group_counts.sort("n", descending=True).row(0, named=True)["_group"]
    already = train_subset.height
    candidates = (
        train_usable
        .filter(pl.col("_group") == biggest_group)
        .join(train_subset.select("id"), on="id", how="anti")
    )
    top_up = candidates.sample(
        n=min(shortfall, candidates.height),
        seed=SUBSET_SEED + 1,
        with_replacement=False,
    )
    train_subset = pl.concat([train_subset, top_up.drop("_group")])
    print(f"  after top-up: {train_subset.height:,} rows")

t_sample = time.time() - t0
print(f"  sampling completed in {t_sample:.1f}s")
print(f"  final subset size : {train_subset.height:,}")
print()

# --- 5. Report class distributions in the subset ----------------------
print("Subset class distributions:")
print("  recommendation_status:")
rec_dist_subset = (
    train_subset.group_by("recommendation_status")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / train_subset.height * 100).round(2).alias("pct"))
    .sort("recommendation_status")
)
print(rec_dist_subset)

print("  rating_band:")
band_dist_subset = (
    train_subset.group_by("rating_band")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / train_subset.height * 100).round(2).alias("pct"))
    .sort("rating_band")
)
print(band_dist_subset)

# --- 6. Write the subset to /content for fast reload ------------------
SUBSET_PARQUET = DRIVE_ROOT / "persian_nlp_pipline" / "artifacts" / "train_subset.parquet"
train_subset.write_parquet(SUBSET_PARQUET, compression="zstd")
print()
print(f"Wrote subset parquet : {SUBSET_PARQUET}")
print(f"  size               : {SUBSET_PARQUET.stat().st_size / 1024**2:.1f} MB")
print(f"  rows               : {train_subset.height:,}")
print()

# --- 7. Confirm val/test stay full (no subsetting) -------------------
assert val_full.height == manifest["counts"]["validation"] if manifest else True
print("Validation and test remain full (no subsetting).")
print()
print("=" * 60)
print("Subset ready. Next: tokenizer + Dataset (cell 3).")
print("=" * 60)

Manifest found — verifying row counts.
  expected train_supervised : 3,578,692
  expected validation       : 913,570
  expected test             : 919,841

Loading split CSVs with polars...
  loaded in 2.8s
  train : 3,578,692
  val   : 913,570
  test  : 919,841
  counts match manifest. OK.

Filtering to rows with both labels present...
  filtered in 0.1s
  usable train rows : 2,950,626  (82.45% of train)

Building 500,000-row stratified subset (seed=2026)...
  Joint group sizes and sample targets:
shape: (9, 3)
┌────────────────────────┬─────────┬──────────┐
│ _group                 ┆ n       ┆ n_sample │
│ ---                    ┆ ---     ┆ ---      │
│ str                    ┆ u32     ┆ i64      │
╞════════════════════════╪═════════╪══════════╡
│ no_idea|high           ┆ 57229   ┆ 9697     │
│ no_idea|low            ┆ 58230   ┆ 9867     │
│ no_idea|medium         ┆ 217102  ┆ 36789    │
│ not_recommended|high   ┆ 6980    ┆ 1182     │
│ not_recommended|low    ┆ 236694  ┆ 40109    │
│ 

In [12]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 3: Tokenizer & Dataset
# ============================================================
# Goal:
#   - load the ParsBERT tokenizer once
#   - define label maps for both tasks (stable order)
#   - define a Dataset that tokenizes on-the-fly
#   - set up dynamic padding with DataCollatorWithPadding
#   - quick sanity check on 5 samples
#   - save the tokenizer to both output dirs in Drive
#
# Writes: REC_OUTPUT_DIR/tokenizer/  and  BAND_OUTPUT_DIR/tokenizer/
# ============================================================

import json
import time
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# --- 1. Load tokenizer (one time) ----------------------------------------
print(f"Loading tokenizer: {MODEL_NAME}")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  loaded in {time.time() - t0:.1f}s")
print(f"  vocab_size : {tokenizer.vocab_size:,}")

# Quick Persian sanity check: a Persian word must map to more than one [UNK].
_test = tokenizer("سلام دنیا این یک تست است", add_special_tokens=False)["input_ids"]
_n_unk = sum(1 for x in _test if x == tokenizer.unk_token_id)
print(f"  sample ids : {_test[:10]}...")
print(f"  unknown tokens in sample: {_n_unk}")
assert _n_unk <= 2, "tokenizer seems to mishandle Persian text"
print()

# --- 2. Label maps (stable order across training, eval, inference) -------
REC_LABEL2ID = {label: i for i, label in enumerate(REC_LABELS)}
REC_ID2LABEL = {i: label for label, i in REC_LABEL2ID.items()}

BAND_LABEL2ID = {label: i for i, label in enumerate(BAND_LABELS)}
BAND_ID2LABEL = {i: label for label, i in BAND_LABEL2ID.items()}

print(f"recommendation label map: {REC_LABEL2ID}")
print(f"rating_band label map   : {BAND_LABEL2ID}")
print()

# --- 3. Dataset class ----------------------------------------------------
class ReviewDataset(Dataset):
    """
    On-the-fly tokenization of Persian review bodies.

    Parameters
    ----------
    df          : polars.DataFrame with columns body_normalized,
                  recommendation_status, rating_band
    tokenizer   : a HuggingFace tokenizer
    max_length  : int
    rec2id      : dict[str, int]
    band2id     : dict[str, int]

    Returns
    -------
    dict with:
        input_ids       : list[int]
        attention_mask  : list[int]
        rec_label       : int
        band_label      : int
    """

    def __init__(self, df, tokenizer, max_length, rec2id, band2id):
        self.texts = df["body_normalized"].fill_null("").to_list()
        self.rec_labels = df["recommendation_status"].to_list()
        self.band_labels = df["rating_band"].to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.rec2id = rec2id
        self.band2id = band2id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding=False,        # padding is done in the collator
            return_tensors=None,  # return Python lists
        )
        enc["rec_label"] = self.rec2id[self.rec_labels[idx]]
        enc["band_label"] = self.band2id[self.band_labels[idx]]
        return enc


# --- 4. Collator: dynamic padding + wrap labels for one task --------------
# We wrap DataCollatorWithPadding to accept our extra label keys
# (rec_label, band_label) and produce a HuggingFace-style batch with
# a single 'labels' field for the task at hand.
def make_collator(tokenizer, task: str):
    """
    task : "recommendation" or "rating_band"
    """
    assert task in ("recommendation", "rating_band")
    base = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    label_key = "rec_label" if task == "recommendation" else "band_label"

    def collate(batch):
        # Pull out the chosen task label before passing to the base collator.
        labels = torch.tensor([b.pop(label_key) for b in batch], dtype=torch.long)
        features = base(batch)                # input_ids, attention_mask
        features["labels"] = labels
        return features

    return collate


# --- 5. Quick sanity check on 5 samples ---------------------------------
print("Sanity check — tokenizing 5 samples from the subset:")
small = train_subset.head(5)
for i in range(small.height):
    text = small["body_normalized"][i]
    rec_lbl = small["recommendation_status"][i]
    band_lbl = small["rating_band"][i]
    enc = tokenizer(
        text, truncation=True, max_length=MAX_LENGTH,
        padding=False, return_tensors=None,
    )
    n_tok = len(enc["input_ids"])
    trunc = " [TRUNCATED]" if n_tok >= MAX_LENGTH else ""
    short = text[:60] + ("..." if len(text) > 60 else "")
    print(f"  text    : {short!r}")
    print(f"  tokens  : {n_tok}{trunc}")
    print(f"  labels  : rec={rec_lbl!r}  band={band_lbl!r}")
    print()

# --- 6. Token-length distribution on 5,000 sampled rows -----------------
print("Token-length distribution on 5,000 sampled rows...")
sample_for_lengths = train_subset.sample(n=5_000, seed=SUBSET_SEED)
lengths = []
t0 = time.time()
for text in sample_for_lengths["body_normalized"].to_list():
    enc = tokenizer(
        text if text is not None else "",
        truncation=True, max_length=MAX_LENGTH,
        padding=False, return_tensors=None,
    )
    lengths.append(len(enc["input_ids"]))
lengths = np.array(lengths)
t_tok = time.time() - t0

print(f"  tokenized 5,000 rows in {t_tok:.1f}s  "
      f"({1000 * t_tok / 5000:.1f} ms/row)")
print(f"  min  : {lengths.min()}")
print(f"  p50  : {int(np.percentile(lengths, 50))}")
print(f"  p90  : {int(np.percentile(lengths, 90))}")
print(f"  p95  : {int(np.percentile(lengths, 95))}")
print(f"  p99  : {int(np.percentile(lengths, 99))}")
print(f"  max  : {lengths.max()}")
print(f"  % at MAX_LENGTH (truncated): "
      f"{100 * (lengths >= MAX_LENGTH).sum() / len(lengths):.2f}%")
print()

# --- 7. Save the tokenizer to both output dirs in Drive -----------------
tok_rec_path  = REC_OUTPUT_DIR / "tokenizer"
tok_band_path = BAND_OUTPUT_DIR / "tokenizer"
tok_rec_path.mkdir(parents=True, exist_ok=True)
tok_band_path.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(tok_rec_path)
tokenizer.save_pretrained(tok_band_path)

print(f"Saved tokenizer to:")
print(f"  {tok_rec_path}")
print(f"  {tok_band_path}")
print()

# --- 8. Ready message ---------------------------------------------------
print("=" * 60)
print("Cell 3 complete: tokenizer + dataset + collator ready.")
print("Next: cell 4 (Task A — BERT fine-tuning for recommendation).")
print("=" * 60)

Loading tokenizer: HooshvareLab/bert-fa-base-uncased
  loaded in 0.8s
  vocab_size : 100,000
  sample ids : [4285, 3810, 2802, 2829, 7623, 2806]...
  unknown tokens in sample: 0

recommendation label map: {'recommended': 0, 'not_recommended': 1, 'no_idea': 2}
rating_band label map   : {'low': 0, 'medium': 1, 'high': 2}

Sanity check — tokenizing 5 samples from the subset:
  text    : 'از قالب بزرگ استفاده\u200cشده و مقداری گشاد بود به خصوص قسمت جلوی...'
  tokens  : 15
  labels  : rec='no_idea'  band='high'

  text    : '3 سال پیش برای صندلی اداره گرفتم. تا حالا که خوب بوده فقط مح...'
  tokens  : 25
  labels  : rec='no_idea'  band='high'

  text    : 'به نظر خوبه'
  tokens  : 5
  labels  : rec='no_idea'  band='high'

  text    : 'در توضیحات رنگ دلخواه را بنویسید'
  tokens  : 8
  labels  : rec='no_idea'  band='high'

  text    : 'خوب هست. ولی به پوسته طرف مقابل هم\u200cبستگی داره.'
  tokens  : 13
  labels  : rec='no_idea'  band='high'

Token-length distribution on 5,000 sampled rows...
 

In [13]:
from pathlib import Path
contents = list(REC_OUTPUT_DIR.iterdir()) if REC_OUTPUT_DIR.exists() else []
print("REC_OUTPUT_DIR contents:", [p.name for p in contents])

REC_OUTPUT_DIR contents: ['tokenizer']


In [19]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4: Task A Training
# ============================================================
# transformers 5.22 compatible: no overwrite_output_dir, no warmup_ratio,
# no logging_dir, no disable_tqdm. Uses warmup_steps and a checkpoint
# helper that handles the "no checkpoint yet" case gracefully.
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model ------------------------------------------------------
print(f"Loading model: {MODEL_NAME}")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets --------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Metrics ---------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 4. TrainingArguments (v5 compatible) ------------------------------
total_optimizer_steps = (len(train_ds_rec) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
print(f"  total steps: {total_optimizer_steps:,}  warmup: {warmup_steps:,}")

training_args_rec = TrainingArguments(
    output_dir=str(REC_OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 5. Trainer --------------------------------------------------------
trainer_rec = Trainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

# --- 6. Train (resume if checkpoint exists, else start fresh) ----------
def latest_checkpoint(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    ckpts = [p for p in output_dir.iterdir()
             if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt = latest_checkpoint(REC_OUTPUT_DIR)
if ckpt is None:
    print("No checkpoint found — starting fresh.")
else:
    print(f"Resuming from checkpoint: {ckpt}")

print("Starting training...")
t0 = time.time()
train_result = trainer_rec.train(resume_from_checkpoint=ckpt)
print(f"Finished in {(time.time() - t0)/60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

Loading model: HooshvareLab/bert-fa-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

  total params: 162,843,651
  train: 500,000  val: 780,620
  total steps: 31,250  warmup: 3,125
No checkpoint found — starting fresh.
Starting training...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,0.756352,0.362696,0.858666,0.654328
4000,0.712524,0.353614,0.863110,0.676738
6000,0.752653,0.355130,0.862801,0.670378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Finished in 55.1 min
  global_step: 6000
  train_loss : 0.7807
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-4000
  best metric    : 0.6767


In [17]:
# Check if trainer_rec exists and its state
try:
    print(f"trainer_rec defined: True")
    print(f"  global_step : {trainer_rec.state.global_step}")
    print(f"  log_history : {len(trainer_rec.state.log_history)} entries")
    print(f"  train_dataset size : {len(trainer_rec.train_dataset):,}")
    print(f"  args.max_steps : {trainer_rec.args.max_steps}")
    print(f"  args.num_train_epochs : {trainer_rec.args.num_train_epochs}")
    print(f"  args.fp16 : {trainer_rec.args.fp16}")
    print(f"  args.eval_strategy : {trainer_rec.args.eval_strategy}")
except NameError:
    print("trainer_rec is NOT defined — error occurred before Trainer.__init__")

trainer_rec defined: True
trainer_rec is NOT defined — error occurred before Trainer.__init__


In [20]:
from pathlib import Path
print("REC_OUTPUT_DIR contents:")
for p in sorted(REC_OUTPUT_DIR.iterdir()):
    if p.is_dir():
        size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
        print(f"  {p.name}/  ({size:.2f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

REC_OUTPUT_DIR contents:
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (1.82 GB)
  tokenizer/  (0.00 GB)


In [22]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4 v3 (FINAL)
# ============================================================
# Resume with WeightedTrainer + LR separate, from a fine-tuned
# model (checkpoint-6000) but with a FRESH optimizer.
# ============================================================

import time
import os
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model from checkpoint-6000 --------------------------------
ckpt_6000 = REC_OUTPUT_DIR / "checkpoint-6000"
assert ckpt_6000.exists(), f"checkpoint-6000 not found at {ckpt_6000}"

# Remove optimizer/scheduler state (mismatch with our 2-group optimizer)
for fname in ["optimizer.pt", "scheduler.pt"]:
    fpath = ckpt_6000 / fname
    if fpath.exists():
        fpath.unlink()
        print(f"Removed: {fpath.name}")

print(f"Loading model from checkpoint-6000...")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    str(ckpt_6000),
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets ------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Class weights --------------------------------------------------
counts = train_subset["recommendation_status"].value_counts()
count_map = {row["recommendation_status"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(REC_LABELS)
class_weights = torch.tensor(
    [N / (n_classes * count_map[label]) for label in REC_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

print("\nClass weights:")
for label, w in zip(REC_LABELS, class_weights.tolist()):
    print(f"  {label:<18}  count={count_map[label]:>7,}  weight={w:.3f}")

# --- 4. WeightedTrainer ------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with 2 param groups ---------------------------------
HEAD_LR = 1e-4
head_params = list(model_rec.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_rec.parameters() if id(p) not in head_param_ids]

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])
print(f"\nOptimizer: AdamW with 2 param groups")
print(f"  encoder : {len(encoder_params)} tensors, lr={LEARNING_RATE}")
print(f"  head    : {len(head_params)} tensors, lr={HEAD_LR}")

# --- 6. Metrics -------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 7. TrainingArguments with max_steps (fresh optimizer) ------------
# max_steps=25,000 = roughly what remains to complete 2 epochs
# (31,250 total - 6,000 already done ≈ 25,250). We use 25,000.
training_args_rec = TrainingArguments(
    output_dir=str(REC_OUTPUT_DIR),
    max_steps=25_000,                       # ← instead of num_train_epochs
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=2_500,                     # 10% of 25,000
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 8. Trainer -------------------------------------------------------
trainer_rec = WeightedTrainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- 9. Train from fresh optimizer (no resume_from_checkpoint) --------
# The model is already fine-tuned (loaded from checkpoint-6000),
# but the optimizer/scheduler start fresh. No global_step mismatch.
print(f"\nStarting training with fresh optimizer (max_steps=25,000)...")
t0 = time.time()
train_result = trainer_rec.train()       # ← NO resume_from_checkpoint
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

# --- 10. List checkpoints --------------------------------------------
ckpts = sorted(
    [p for p in REC_OUTPUT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in REC_OUTPUT_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

Removed: optimizer.pt
Removed: scheduler.pt
Loading model from checkpoint-6000...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  total params: 162,843,651
  train: 500,000  val: 780,620

Class weights:
  recommended         count=395,948  weight=0.421
  not_recommended     count= 47,699  weight=3.494
  no_idea             count= 56,353  weight=2.958

Optimizer: AdamW with 2 param groups
  encoder : 199 tensors, lr=2e-05
  head    : 2 tensors, lr=0.0001

Starting training with fresh optimizer (max_steps=25,000)...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,1.155707,0.620620,0.832515,0.693445
4000,0.960126,0.695440,0.836748,0.684249


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,1.155707,0.620620,0.832515,0.693445
4000,0.960126,0.695440,0.836748,0.684249
6000,1.035719,0.802278,0.847447,0.692413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 59.5 min
  global_step: 6000
  train_loss : 1.0501
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-2000
  best metric    : 0.6934

Checkpoints in REC_OUTPUT_DIR (3):
  checkpoint-2000  (1.82 GB)
  checkpoint-4000  (1.82 GB)
  checkpoint-6000  (1.82 GB)


In [23]:
import os
print(f"REC_OUTPUT_DIR: {REC_OUTPUT_DIR}")
print(f"Contents:")
for p in sorted(REC_OUTPUT_DIR.iterdir()):
    if p.is_dir():
        size_gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
        print(f"  {p.name}/  ({size_gb:.2f} GB)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1024**2:.1f} MB)")

REC_OUTPUT_DIR: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation
Contents:
  checkpoint-2000/  (1.82 GB)
  checkpoint-4000/  (1.82 GB)
  checkpoint-6000/  (1.82 GB)
  tokenizer/  (0.00 GB)


In [24]:
print("\nFinal Task A results (v3, with class_weights):")
print(f"  best F1 Macro: {trainer_rec.state.best_metric:.4f}")
print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")


Final Task A results (v3, with class_weights):
  best F1 Macro: 0.6934
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/checkpoint-2000


In [25]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 4 v5 (FULL 1 EPOCH)
# ============================================================
# Complete one full epoch (15,625 steps) from checkpoint-2000 (best F1).
# - learning_rate=1e-5 (lower, to reduce overfitting)
# - weight_decay=0.05 (higher, more regularization)
# - NO early stopping (to complete the epoch)
# - max_steps=15,625 (one full epoch)
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load from best checkpoint -------------------------------------
best_ckpt = REC_OUTPUT_DIR / "checkpoint-2000"
assert best_ckpt.exists(), f"checkpoint-2000 not found: {best_ckpt}"

# Remove optimizer/scheduler (mismatch with our optimizer)
for fname in ["optimizer.pt", "scheduler.pt"]:
    fpath = best_ckpt / fname
    if fpath.exists():
        fpath.unlink()
        print(f"Removed: {fpath.name}")

print(f"Loading from {best_ckpt.name}...")
model_rec = AutoModelForSequenceClassification.from_pretrained(
    str(best_ckpt),
    num_labels=len(REC_LABELS),
    id2label=REC_ID2LABEL,
    label2id=REC_LABEL2ID,
)
print(f"  total params: {sum(p.numel() for p in model_rec.parameters()):,}")

# --- 2. Datasets ------------------------------------------------------
train_ds_rec = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_a = val_full.filter(pl.col("recommendation_status").is_not_null())
val_ds_rec = ReviewDataset(val_full_task_a, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_rec = make_collator(tokenizer, task="recommendation")
print(f"  train: {len(train_ds_rec):,}  val: {len(val_ds_rec):,}")

# --- 3. Class weights -------------------------------------------------
counts = train_subset["recommendation_status"].value_counts()
count_map = {row["recommendation_status"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(REC_LABELS)
class_weights = torch.tensor(
    [N / (n_classes * count_map[label]) for label in REC_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

# --- 4. WeightedTrainer ----------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with lower LR ---------------------------------------
HEAD_LR = 5e-5   # from 1e-4
ENCODER_LR = 1e-5   # from 2e-5

head_params = list(model_rec.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_rec.parameters() if id(p) not in head_param_ids]

optimizer = torch.optim.AdamW([
    {"params": encoder_params, "lr": ENCODER_LR, "weight_decay": 0.05},
    {"params": head_params,    "lr": HEAD_LR,    "weight_decay": 0.05},
])
print(f"\nOptimizer: AdamW")
print(f"  encoder : lr={ENCODER_LR}, weight_decay=0.05")
print(f"  head    : lr={HEAD_LR}, weight_decay=0.05")

# --- 6. Metrics -------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# --- 7. TrainingArguments (full 1 epoch, no early stopping) -----------
OUTPUT_DIR_V5 = REC_OUTPUT_DIR / "v5_full_epoch"
OUTPUT_DIR_V5.mkdir(parents=True, exist_ok=True)

training_args_rec = TrainingArguments(
    output_dir=str(OUTPUT_DIR_V5),
    max_steps=15_625,                   # ← یک epoch کامل
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=ENCODER_LR,
    weight_decay=0.05,
    warmup_steps=1_562,                 # 10% of 15,625
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)

# --- 8. Trainer (NO early stopping — we want a full epoch) -----------
trainer_rec = WeightedTrainer(
    model=model_rec,
    args=training_args_rec,
    train_dataset=train_ds_rec,
    eval_dataset=val_ds_rec,
    data_collator=collate_rec,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    optimizers=(optimizer, None),
    # NO EarlyStoppingCallback — we want the full 15,625 steps
)

# --- 9. Train ---------------------------------------------------------
print(f"\nTraining for one full epoch (15,625 steps)...")
t0 = time.time()
train_result = trainer_rec.train()
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result.global_step}")
print(f"  train_loss : {train_result.training_loss:.4f}")
if trainer_rec.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_rec.state.best_model_checkpoint}")
if trainer_rec.state.best_metric is not None:
    print(f"  best metric    : {trainer_rec.state.best_metric:.4f}")

# --- 10. List checkpoints --------------------------------------------
ckpts = sorted(
    [p for p in OUTPUT_DIR_V5.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in v5_full_epoch ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

Removed: optimizer.pt
Removed: scheduler.pt
Loading from checkpoint-2000...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  total params: 162,843,651
  train: 500,000  val: 780,620

Optimizer: AdamW
  encoder : lr=1e-05, weight_decay=0.05
  head    : lr=5e-05, weight_decay=0.05

Training for one full epoch (15,625 steps)...


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
2000,0.619517,0.937088,0.833716,0.683089
4000,0.947418,0.670847,0.837306,0.694896
6000,0.907503,0.799810,0.836003,0.691386
8000,1.142401,0.595697,0.801007,0.683265
10000,1.191680,0.595963,0.835289,0.701713
12000,1.214194,0.582351,0.823804,0.698499
14000,1.198849,0.579589,0.832706,0.701040
15625,1.168670,0.581749,0.826356,0.699014


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished in 159.2 min
  global_step: 15625
  train_loss : 1.0763
  best checkpoint: /content/drive/MyDrive/persian_nlp_pipline/artifacts/bert_recommendation/v5_full_epoch/checkpoint-10000
  best metric    : 0.7017

Checkpoints in v5_full_epoch (3):
  checkpoint-10000  (1.82 GB)
  checkpoint-14000  (1.82 GB)
  checkpoint-15625  (1.82 GB)


In [26]:
# Check Colab GPU quota and session time
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check how long this session has been running
import time
from datetime import datetime
import psutil
boot = datetime.fromtimestamp(psutil.boot_time())
uptime = datetime.now() - boot
print(f"\nSession uptime: {uptime}")

Wed Nov 11 17:47:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             33W /   70W |    4565MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Task B Training

In [ ]:
# ============================================================
# 04_transfer_learning_finetuning.ipynb — Cell 5: Task B Training
# ============================================================
# Fine-tune ParsBERT for rating_band (low/medium/high).
#
# Configuration (adapted from Task A lessons):
#   - start from MODEL_NAME (fresh, not from Task A checkpoint)
#   - encoder lr=2e-5, head lr=1e-4
#   - class_weights from training distribution (imbalance 3.8:1)
#   - fp16, batch 16 x grad_accum 2 = effective 32
#   - max_steps=15,625 (one epoch)
#   - EarlyStoppingCallback(patience=2)
#   - resume_from_checkpoint via helper (safe in Colab)
#
# Writes checkpoint-XXXX/ every 2000 steps into BAND_OUTPUT_DIR (Drive).
# ============================================================

import time
import numpy as np
import polars as pl
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# --- 1. Load model (fresh, not from Task A) ----------------------------
print(f"Loading model: {MODEL_NAME}")
t0 = time.time()
model_band = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(BAND_LABELS),
    id2label=BAND_ID2LABEL,
    label2id=BAND_LABEL2ID,
)
print(f"  loaded in {time.time() - t0:.1f}s")
total_params     = sum(p.numel() for p in model_band.parameters())
trainable_params = sum(p.numel() for p in model_band.parameters() if p.requires_grad)
print(f"  total params     : {total_params:,}")
print(f"  trainable params : {trainable_params:,}")
print()

# --- 2. Datasets ------------------------------------------------------
# train_subset already has both labels present.
# val_full: filter to valid rating_band (exclude 1 invalid row)
train_ds_band = ReviewDataset(train_subset, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
val_full_task_b = val_full.filter(
    pl.col("is_valid_rating_band") & pl.col("rating_band").is_not_null()
)
val_ds_band = ReviewDataset(val_full_task_b, tokenizer, MAX_LENGTH, REC_LABEL2ID, BAND_LABEL2ID)
collate_band = make_collator(tokenizer, task="rating_band")

print(f"  train rows (Task B) : {len(train_ds_band):,}")
print(f"  val rows   (Task B) : {len(val_ds_band):,}")
print()

# --- 3. Class weights (inverse frequency) -----------------------------
counts = train_subset["rating_band"].value_counts()
count_map = {row["rating_band"]: row["count"] for row in counts.iter_rows(named=True)}
N = sum(count_map.values())
n_classes = len(BAND_LABELS)
class_weights_band = torch.tensor(
    [N / (n_classes * count_map[label]) for label in BAND_LABELS],
    dtype=torch.float,
).to("cuda" if torch.cuda.is_available() else "cpu")

print("Class weights (inverse frequency):")
for label, w in zip(BAND_LABELS, class_weights_band.tolist()):
    print(f"  {label:<8}  count={count_map[label]:>7,}  weight={w:.3f}")
print()

# --- 4. WeightedTrainer (same as Task A) ------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- 5. Optimizer with separate LR for encoder and head ---------------
HEAD_LR = 1e-4
head_params = list(model_band.classifier.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in model_band.parameters() if id(p) not in head_param_ids]

optimizer_band = torch.optim.AdamW([
    {"params": encoder_params, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    {"params": head_params,    "lr": HEAD_LR,      "weight_decay": WEIGHT_DECAY},
])
print(f"Optimizer: AdamW")
print(f"  encoder : {len(encoder_params)} tensors, lr={LEARNING_RATE}")
print(f"  head    : {len(head_params)} tensors, lr={HEAD_LR}")
print()

# --- 6. Metrics (add ordinal distance error) --------------------------
# BAND_ORDINAL = {"low": 0, "medium": 1, "high": 2}
def compute_metrics_band(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    # Ordinal distance error: mean |true_ordinal - pred_ordinal|
    # labels here are integers (0=low, 1=medium, 2=high) from the model's label2id
    ord_err = float(np.mean(np.abs(labels - preds)))

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "ordinal_distance_error": ord_err,
    }

# --- 7. TrainingArguments (one epoch, v5-compatible) ------------------
BAND_TOTAL_STEPS = (len(train_ds_band) // (BATCH_SIZE * GRAD_ACCUM)) * 1   # 1 epoch
warmup_steps_band = max(1, int(BAND_TOTAL_STEPS * WARMUP_RATIO))

training_args_band = TrainingArguments(
    output_dir=str(BAND_OUTPUT_DIR),
    max_steps=BAND_TOTAL_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps_band,
    lr_scheduler_type="linear",
    fp16=FP16,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=TRAIN_SEED,
    data_seed=TRAIN_SEED,
)
print(f"Training schedule:")
print(f"  max_steps      : {BAND_TOTAL_STEPS:,}  (1 epoch)")
print(f"  warmup_steps   : {warmup_steps_band:,}")
print(f"  eval every     : {EVAL_STEPS} steps")
print(f"  save every     : {SAVE_STEPS} steps")
print()

# --- 8. Trainer -------------------------------------------------------
trainer_band = WeightedTrainer(
    model=model_band,
    args=training_args_band,
    train_dataset=train_ds_band,
    eval_dataset=val_ds_band,
    data_collator=collate_band,
    compute_metrics=compute_metrics_band,
    class_weights=class_weights_band,
    optimizers=(optimizer_band, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# --- 9. Resume if checkpoint exists, else start fresh ----------------
def latest_checkpoint(output_dir: Path) -> str | None:
    if not output_dir.exists():
        return None
    ckpts = [p for p in output_dir.iterdir()
             if p.is_dir() and p.name.startswith("checkpoint-")]
    if not ckpts:
        return None
    ckpts.sort(key=lambda p: int(p.name.split("-")[1]))
    return str(ckpts[-1])

ckpt_band = latest_checkpoint(BAND_OUTPUT_DIR)
if ckpt_band is None:
    print("No checkpoint found — starting fresh.")
else:
    print(f"Resuming from checkpoint: {ckpt_band}")

print("\nStarting Task B training...")
t0 = time.time()
train_result_band = trainer_band.train(resume_from_checkpoint=ckpt_band)
t_train = time.time() - t0

print(f"\nFinished in {t_train / 60:.1f} min")
print(f"  global_step: {train_result_band.global_step}")
print(f"  train_loss : {train_result_band.training_loss:.4f}")
if trainer_band.state.best_model_checkpoint:
    print(f"  best checkpoint: {trainer_band.state.best_model_checkpoint}")
if trainer_band.state.best_metric is not None:
    print(f"  best metric    : {trainer_band.state.best_metric:.4f}")

# --- 10. List checkpoints ---------------------------------------------
ckpts = sorted(
    [p for p in BAND_OUTPUT_DIR.iterdir() if p.is_dir() and p.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1])
)
print(f"\nCheckpoints in BAND_OUTPUT_DIR ({len(ckpts)}):")
for c in ckpts:
    size_gb = sum(f.stat().st_size for f in c.rglob("*") if f.is_file()) / 1024**3
    print(f"  {c.name}  ({size_gb:.2f} GB)")

print("\n" + "=" * 60)
print("Task B training complete.")
print("Next: cell 6 (evaluation on test + confusion matrix).")
print("=" * 60)

Loading model: HooshvareLab/bert-fa-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

  loaded in 1.0s
  total params     : 162,843,651
  trainable params : 162,843,651

  train rows (Task B) : 500,000
  val rows   (Task B) : 913,570

Class weights (inverse frequency):
  low       count= 93,678  weight=1.779
  medium    count= 89,797  weight=1.856
  high      count=316,525  weight=0.527

Optimizer: AdamW
  encoder : 199 tensors, lr=2e-05
  head    : 2 tensors, lr=0.0001

Training schedule:
  max_steps      : 15,625  (1 epoch)
  warmup_steps   : 1,562
  eval every     : 2000 steps
  save every     : 2000 steps

No checkpoint found — starting fresh.

Starting Task B training...


Step,Training Loss,Validation Loss
